# Basic operators with multiproc/multithread

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import concurrent.futures as mt
import multiprocessing as mp
import numpy as np
import pylops

## Matmat/Rmatmat

In [2]:
M = np.random.normal(0, 1, (2000, 1000))

Mop = pylops.MatrixMult(M)

X = np.ones((1000, 160))

Y = Mop.matmat(X)
Y.shape

(2000, 160)

In [3]:
poolt = mt.ThreadPoolExecutor(max_workers=4)
poolm = mp.Pool(processes=4)

Ymt = Mop.matmat(X, pool=poolt)
Ymp = Mop.matmat(X, pool=poolm)

np.allclose(Y, Ymt), np.allclose(Y, Ymp)

(True, True)

In [4]:
Xadj = Mop.rmatmat(Y)
Xadjmt = Mop.rmatmat(Y, pool=poolt)
Xadjmp = Mop.rmatmat(Y, pool=poolm)

np.allclose(Xadj, Xadjmt), np.allclose(Xadj, Xadjmp)

(True, True)

In [5]:
%timeit -n 5 -r 2 Mop.matmat(X)
%timeit -n 5 -r 2 Mop.matmat(X, pool=poolt)
%timeit -n 5 -r 2 Mop.matmat(X, pool=poolm)

33.1 ms ± 232 μs per loop (mean ± std. dev. of 2 runs, 5 loops each)
22.4 ms ± 5.49 ms per loop (mean ± std. dev. of 2 runs, 5 loops each)
259 ms ± 18.4 ms per loop (mean ± std. dev. of 2 runs, 5 loops each)


In [6]:
%timeit -n 5 -r 2 Mop.rmatmat(Y)
%timeit -n 5 -r 2 Mop.rmatmat(Y, pool=poolt)
%timeit -n 5 -r 2 Mop.rmatmat(Y, pool=poolm)

33.6 ms ± 41.4 μs per loop (mean ± std. dev. of 2 runs, 5 loops each)
13.8 ms ± 674 μs per loop (mean ± std. dev. of 2 runs, 5 loops each)
223 ms ± 2.9 ms per loop (mean ± std. dev. of 2 runs, 5 loops each)


## Kronecker

In [13]:
np.random.seed(10)

ny, nx = 2000, 1000
G1 = np.random.normal(0, 10, (ny, nx))
G2 = np.random.normal(0, 10, (ny, nx))
x = np.ones(nx ** 2)

Kop = pylops.Kronecker(
    pylops.MatrixMult(G1),
    pylops.MatrixMult(G2)
)

y = Kop * x
xadj = Kop.H * y

In [14]:
K1op = pylops.Kronecker(
    pylops.MatrixMult(G1),
    pylops.MatrixMult(G2),
    nproc=4,
    parallel_kind="multithread",

)

y1 = K1op * x
x1adj = K1op.H * y

np.allclose(y, y1), np.allclose(xadj, x1adj)

(True, True)

In [15]:
%timeit -n 5 -r 2 Kop * x
%timeit -n 5 -r 2 K1op * x

%timeit -n 5 -r 2 Kop.H * y
%timeit -n 5 -r 2 K1op.H * y

395 ms ± 68.9 ms per loop (mean ± std. dev. of 2 runs, 5 loops each)
192 ms ± 1.22 ms per loop (mean ± std. dev. of 2 runs, 5 loops each)
345 ms ± 7.61 ms per loop (mean ± std. dev. of 2 runs, 5 loops each)
208 ms ± 4.37 ms per loop (mean ± std. dev. of 2 runs, 5 loops each)
